# Dependencies


In [9]:
# pip install ultralytics opencv-python matplotlib pandas scikit-image --no-cache

## Libraries

In [10]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

### Checking CUDA Avaibility

In [11]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

torch.cuda.set_per_process_memory_fraction(
    6.5 / 8.0,
    device=0
)

True
1
NVIDIA GeForce RTX 5050 Laptop GPU


# Google Drive for Dataset and Research

In [12]:
# from google.colab import drive

# drive.mount('/content/drive')

# Builiding Models

## Create dataset.yaml

In [13]:
print(os.getcwd())

yaml_content = """
path: dataset

train: images/train
val: images/val

names:
  0: droplet
"""

# with open('/content/drive/MyDrive/Project/DropletDetection/dataset/dataset.yaml', 'w') as f:
#     f.write(yaml_content)
with open('dataset/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print('dataset.yaml created successfully')

## Verify Dataset
# train_path = '/content/drive/MyDrive/Project/DropletDetection/dataset/images/train'
train_path = 'dataset/images/train'

print('Training Images:', len(os.listdir(train_path)))

/home/adli-ubuntu/Documents/Project/DropletDetection
dataset.yaml created successfully
Training Images: 250


## Sample Image View

In [14]:
# sample_image = os.path.join(train_path, os.listdir(train_path)[0])

# image = cv2.imread(sample_image)
# image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# plt.figure(figsize=(8,8))
# plt.imshow(image)
# plt.title('Sample WSP Image')
# plt.axis('off')
# plt.show()

## OpenCV Preprocessing

In [15]:
## Preprocessing Function
def preprocess_wsp(image_path):

    image = cv2.imread(image_path)

    # Resize optional
    image = cv2.resize(image, (768, 768))

    # Convert to HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Blue droplet threshold
    lower_blue = np.array([90, 50, 50])
    upper_blue = np.array([140, 255, 255])

    mask = cv2.inRange(hsv, lower_blue, upper_blue)

    # Median blur
    mask = cv2.medianBlur(mask, 3)

    # Morphology cleanup
    kernel = np.ones((3,3), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    return image, mask

# ## Testing Function
# image, mask = preprocess_wsp(sample_image)

# plt.figure(figsize=(8,4))

# plt.subplot(1,2,1)
# plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
# plt.title('Original Image')
# plt.axis('off')

# plt.subplot(1,2,2)
# plt.imshow(mask, cmap='gray')
# plt.title('Preprocessed Mask')
# plt.axis('off')

# plt.show()

# YOLO

## Train and Validate Model

In [16]:

### Recommend Order
## Best : yolov8s-seg + imgz 1024
## Fast : yolov8n-seg + imgz 768
## high accuracy : yolov8m-seg + imgz 1024

## Reseting GPU Memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Model Training
MODEL_NAME = "yolov8n-seg.pt" ## If GPU is Out of Memory
# MODEL_NAME = "yolov8s-seg.pt" ## Fast
# MODEL_NAME = "yolov8s-seg.pt" ## Recommend
# MODEL_NAME = "yolov8m-seg.pt" ## High Accuracy


## Parameters
IMG_SIZE = 512 ## if GPU is Out of Memory
# IMG_SIZE = 768 ## Fast and recommend
# IMG_SIZE = 1024 ## High Accuracy


## Batch Size
BATCH_SIZE = 2 ## 
# BATCH_SIZE = 4 ## 
# BATCH_SIZE = 8 ## 


## Epochs and Patience
EPOCHS = 120
PATIENCE = 10

model = YOLO(MODEL_NAME) ## Fast

results = model.train(
    data='dataset/dataset.yaml',

    epochs=EPOCHS,
    patience=10,

    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,

    optimizer="AdamW",
    lr0=0.0005,
    weight_decay=0.0005,

    mosaic=0.3,
    close_mosaic=15,

    overlap_mask=True,
    mask_ratio=1,

    amp=True,
    cache=False,
    workers=0, # Fast and recommend, set to 0 for Windows to avoid multiprocessing issues

    project="DropletDetection_YoloV8",

    name=f"{MODEL_NAME.replace('.pt','')}_{IMG_SIZE}_bs{BATCH_SIZE}",

    exist_ok=True
)

metrics = model.val()

print(metrics)

New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.50 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 7708MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=1, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=0.3, mul

OutOfMemoryError: CUDA out of memory. Tried to allocate 1020.00 MiB. GPU 0 has a total capacity of 7.53 GiB of which 1.86 GiB is free. Including non-PyTorch memory, this process has 5.64 GiB memory in use. 6.12 GiB allowed; Of the allocated memory 3.95 GiB is allocated by PyTorch, and 1.51 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## Runnging Prediciton

In [ ]:
## Selecting the best model for inference
best_model = YOLO(
    "YOLOv8/wsp_yolov8_seg/weights/best.pt"
)

## READ ORIGINAL IMAGE
original_image = cv2.imread(sample_image)

## Convert BGR -> RGB
original_rgb = cv2.cvtColor(
    original_image,
    cv2.COLOR_BGR2RGB
)

## Run Prediction
results = best_model.predict(
    source=sample_image,

    imgsz=768,
    conf=0.25,

    retina_masks=True,

    save=True,
    save_txt=True,
    save_conf=True
)

## SIDE-BY-SIDE VISUALIZATION
pred_image = results[0].plot()

pred_rgb = cv2.cvtColor(
    pred_image,
    cv2.COLOR_BGR2RGB
)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

## Original
axes[0].imshow(original_rgb)
axes[0].set_title("Original Image")
axes[0].axis("off")

## Prediction
axes[1].imshow(pred_rgb)
axes[1].set_title("YOLOv8-seg Prediction")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Coverage Estimation and Droplet Counting

In [ ]:
## Coverage Percentage
masks = results[0].masks.data.cpu().numpy()
combined_mask = np.any(masks, axis=0)
binary_mask = combined_mask.astype(np.uint8)*255
droplet_pixels = np.sum(binary_mask == 255)
total_pixels = binary_mask.size
coverage = (droplet_pixels / total_pixels) * 100

print(f'Coverage Percentage: {coverage:.2f}%')

## Droplet Counting
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask)
actual_droplets = num_labels - 1

print('Droplet Count:', actual_droplets)

# Export Result

In [ ]:
# results_df = pd.DataFrame({
#     'coverage_percent': [coverage],
#     'droplet_count': [actual_droplets],
#     'mean_diameter_pixels': [mean_diameter]
# })

# csv_path = '/content/drive/MyDrive/Project/DropletDetection/DD_Training/results.csv'

# results_df.to_csv(csv_path, index=False)

# print('Results exported to:', csv_path)

# Save Binary Mask

In [ ]:
# mask_path = '/content/drive/MyDrive/Project/DropletDetection/DD_Training/binary_mask.png'

# cv2.imwrite(mask_path, binary_mask)

# print('Mask saved to:', mask_path)

# Batch Processing

In [ ]:
# input_folder = '/content/drive/MyDrive/WSP_Dataset/images/test'

# all_results = []

# for filename in os.listdir(input_folder):

#     image_path = os.path.join(input_folder, filename)

#     results = best_model.predict(image_path, conf=0.25)

#     masks = results[0].masks.data.cpu().numpy()

#     combined_mask = np.any(masks, axis=0)

#     binary_mask = combined_mask.astype(np.uint8) * 255

#     droplet_pixels = np.sum(binary_mask == 255)

#     total_pixels = binary_mask.size

#     coverage = (droplet_pixels / total_pixels) * 100

#     num_labels, labels = cv2.connectedComponents(binary_mask)

#     droplet_count = num_labels - 1

#     all_results.append({
#         'image': filename,
#         'coverage_percent': coverage,
#         'droplet_count': droplet_count
#     })

# batch_df = pd.DataFrame(all_results)

# batch_csv = '/content/drive/MyDrive/WSP_Training/batch_results.csv'

# batch_df.to_csv(batch_csv, index=False)

# print('Batch processing completed')